# Monitor de modificaciones CSV → Firestore

Instalar las dependencias de [requirements.txt](./requirements.txt) en un entorno virtual y revisar [SECURITY.md](./SECURITY.md).

La siguiente celda escribe en Firestore y permanece ejecutándose. Iniciar Jupyter desde la raíz del repositorio; usa `Dashboard/csv` o la variable `PI_CSV_DIR`. Solo procesa eventos de modificación de CSV en esa carpeta, sin recorrer subcarpetas ni hacer una carga inicial.


In [ ]:
import os
import pandas as pd
import firebase_admin
from firebase_admin import credentials, firestore
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler
import time

# Configurar Firebase con las credenciales
cred = credentials.ApplicationDefault()
try:
    firebase_admin.get_app()
except ValueError:
    firebase_admin.initialize_app(cred)
db = firestore.client()

# Ruta a la carpeta donde están los archivos CSV
csv_folder = os.environ.get("PI_CSV_DIR", os.path.join(os.getcwd(), "Dashboard", "csv"))

# Función para subir datos a Firebase
def upload_to_firebase(csv_file):
    full_csv_path = os.path.join(csv_folder, csv_file)

    try:
        df = pd.read_csv(full_csv_path)

        # Crear o referenciar la colección en Firestore
        collection_name = os.path.splitext(csv_file)[0]
        collection_ref = db.collection(collection_name)

        # Subir los datos a Firestore
        for index, row in df.iterrows():
            doc_ref = collection_ref.document(str(index))
            doc_ref.set(row.to_dict())

        print(f"Datos de {csv_file} actualizados en Firebase.")

    except Exception as e:
        print(f"Error al actualizar {csv_file}: {e}")

# Clase que maneja eventos de cambio en archivos CSV
class CSVHandler(FileSystemEventHandler):
    def on_modified(self, event):
        if event.is_directory:
            return
        if event.src_path.endswith(".csv"):
            csv_file = os.path.basename(event.src_path)
            print(f"Detectado cambio en {csv_file}, actualizando en Firebase...")
            upload_to_firebase(csv_file)

# Configurar el monitoreo de la carpeta
event_handler = CSVHandler()
observer = Observer()
observer.schedule(event_handler, path=csv_folder, recursive=False)

print("Monitoreo iniciado... (Presiona Ctrl+C para detener)")
observer.start()

# Mantener el script en ejecución
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    observer.stop()
    print("Monitoreo detenido.")

observer.join()


## Ejecución y límites

Para detener el monitor, interrumpir la celda de Jupyter: el código captura `KeyboardInterrupt`, detiene el observador y espera su cierre.

No se incluye un ejecutable `monitor_csv.py` ni una tarea programada. El notebook no es un servicio instalado. Los eventos repetidos pueden recargar el mismo archivo; no hay control de concurrencia, espera a escrituras completas ni sincronización con Power BI.

Consultar el [mapa técnico](./docs/ARQUITECTURA.md) antes de ampliar la automatización.
